In [3]:
# imports, reading and cleaning
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.impute import SimpleImputer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.feature_selection import mutual_info_classif
from sklearn.datasets import make_classification

df = pd.read_csv("hotel_bookings.csv")
print("Dataset shape:", df.shape)
print(df.head())
df['country'] = df['country'].fillna('Unknown')
df['agent'] = df['agent'].fillna(0)
df['company'] = df['company'].fillna(0)
df['children'] = df['children'].fillna(0)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# task 1
print("\n--- TASK 1: BASELINE ---")

y = df['is_canceled']
X = df.drop(columns=['is_canceled'])

X = pd.get_dummies(X, drop_first=True)
X = X.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
# task 2
print("\n--- TASK 2: DIMENSIONALITY ---")

for dim in [2, 10, 50, 200]:
    X_syn, _ = make_classification(
        n_features=dim,
        n_samples=1000,
        n_informative=2,
        n_redundant=0,
        n_repeated=0,
        random_state=42
    )
    
    distances = []
    for _ in range(100):
        a, b = X_syn[np.random.randint(0,1000)], X_syn[np.random.randint(0,1000)]
        distances.append(np.linalg.norm(a-b))
    
    plt.hist(distances, bins=30)
    plt.title(f"Distance Distribution ({dim} features)")
    plt.show()


In [ ]:
# task 3
print("\n--- TASK 3: NUMERIC PREPROCESSING ---")

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols.remove('is_canceled')

df_num = df[numeric_cols].copy()

df_num['lead_time_bin'] = pd.qcut(df_num['lead_time'], q=4, duplicates='drop')
df_num['high_adr'] = (df_num['adr'] > df_num['adr'].median()).astype(int)

for name, scaler in {
    "MinMax": MinMaxScaler(),
    "Standard": StandardScaler(),
    "Robust": RobustScaler()
}.items():
    scaled = scaler.fit_transform(df_num[['adr']])
    plt.hist(scaled, bins=30)
    plt.title(f"{name} Scaler - ADR")
    plt.show()

In [ ]:
# task 4
print("\n--- TASK 4: KNN SCALING ---")

X_knn = df[numeric_cols].fillna(0)
y_knn = df['is_canceled']

X_train, X_test, y_train, y_test = train_test_split(X_knn, y_knn, test_size=0.2, random_state=42)

knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
print("KNN no scaling:", accuracy_score(y_test, knn.predict(X_test)))

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

knn.fit(X_train_s, y_train)
print("KNN StandardScaler:", accuracy_score(y_test, knn.predict(X_test_s)))

scaler = RobustScaler()
X_train_r = scaler.fit_transform(X_train)
X_test_r = scaler.transform(X_test)

knn.fit(X_train_r, y_train)
print("KNN RobustScaler:", accuracy_score(y_test, knn.predict(X_test_r)))

In [ ]:
# task 5
print("\n--- TASK 5: PIPELINE ---")

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, numeric_cols)
])

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

scores = cross_val_score(pipeline, df[numeric_cols], df['is_canceled'], cv=5)
print("Pipeline CV Score:", scores.mean())

In [ ]:
# task 6
print("\n--- TASK 6: FEATURE EXTRACTION ---")

df_fe = df.copy()

df_fe['arrival_date'] = pd.to_datetime(
    df_fe['arrival_date_year'].astype(str) + '-' +
    df_fe['arrival_date_month'] + '-' +
    df_fe['arrival_date_day_of_month'].astype(str),
    errors='coerce'
)

df_fe['month'] = df_fe['arrival_date'].dt.month
df_fe['weekday'] = df_fe['arrival_date'].dt.weekday
df_fe['is_weekend'] = df_fe['weekday'].isin([5,6]).astype(int)


In [ ]:
# task 7
print("\n--- TASK 7: FEATURE CONSTRUCTION ---")

df_fc = df_fe.copy()

df_fc['total_nights'] = df_fc['stays_in_week_nights'] + df_fc['stays_in_weekend_nights']
df_fc['price_per_person'] = df_fc['adr'] / (df_fc['adults'] + df_fc['children'] + 1)
df_fc['is_family'] = (df_fc['children'] + df_fc['babies'] > 0).astype(int)
df_fc['special_requests_rate'] = df_fc['total_of_special_requests'] / (df_fc['total_nights'] + 1)
df_fc['interaction'] = df_fc['adr'] * df_fc['lead_time']

df_fc['avg_adr_country'] = df_fc.groupby('country')['adr'].transform('mean')

poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(df_fc[['adr', 'lead_time']].fillna(0))

print("Polynomial features shape:", poly_features.shape)

In [ ]:
# task 8
print("\n--- TASK 8: FEATURE IMPORTANCE ---")

df_model = df_fc.fillna(0)

X = df_model.drop(columns=[
    'is_canceled',
    'arrival_date',
    'reservation_status',
    'reservation_status_date'
])

X = pd.get_dummies(X, drop_first=True)

y = df_model['is_canceled']

model = RandomForestClassifier(random_state=42)
model.fit(X, y)

importances = pd.Series(model.feature_importances_, index=X.columns)
print("Top Features:\n", importances.sort_values(ascending=False).head(15))

mi = mutual_info_classif(X, y)
mi_series = pd.Series(mi, index=X.columns)
print("Top MI Features:\n", mi_series.sort_values(ascending=False).head(15))